# 🎙️ Piper TTS: Персональный офлайн-голос для Android

**Инструкция: Нажмите `Среда выполнения` -> `Выполнить все` (Runtime -> Run all).**
Всё остальное выполнится автоматически:
1. Создается изолированная среда **Python 3.10** через `uv` (обход ограничений Colab Python 3.12/3.13).
2. Загружается готовый нарезанный датасет (50 фраз).
3. Скачивается базовый дикторский чекпоинт `ru_RU-dmitri-medium`.
4. Запускается дообучение (Fine-Tuning) на GPU T4.
5. Автоматически экспортируется файл `my_voice.onnx` (~60 МБ) и скачивается на устройство.

## Шаг 1. Подготовка изолированного окружения Python 3.10 и библиотек

In [ ]:
!nvidia-smi

# Системные утилиты
!apt-get update -qq && apt-get install -y -qq espeak-ng ffmpeg build-essential git

# Установка uv и создание Python 3.10 venv
!curl -LsSf https://astral.sh/uv/install.sh | sh
!/root/.local/bin/uv venv --python 3.10 /content/env

# Установка PyTorch 2.1 с поддержкой CUDA 12.1
!/content/env/bin/pip install -q --upgrade pip
!/content/env/bin/pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Установка совместимых версий PyTorch Lightning, Piper и ONNX
!/content/env/bin/pip install -q pytorch-lightning==1.9.5 torchmetrics==0.11.4 piper-phonemize==1.1.0 'cython<3' onnx onnxruntime soundfile librosa piper-tts

# Клонирование репозитория Piper
import os
if not os.path.exists('/content/piper'):
    !git clone --depth 1 https://github.com/rhasspy/piper.git /content/piper

# Сборка C-расширения monotonic_align для VITS
!cd /content/piper/src/python/piper_train/vits/monotonic_align && /content/env/bin/python setup.py build_ext --inplace

# Установка локальных пакетов piper в venv
!/content/env/bin/pip install -q --no-deps -e /content/piper/src/python_run
!/content/env/bin/pip install -q --no-deps -e /content/piper/src/python

# Проверка работоспособности
!/content/env/bin/python -c "import torch, piper_train, piper_phonemize, pytorch_lightning; print('✓ CUDA доступна:', torch.cuda.is_available()); print('✓ Окружение Python 3.10 полностью готово!')"

## Шаг 2. Загрузка вашего датасета и базовой модели

In [ ]:
import os, shutil

# Клонируем или обновляем репозиторий с аудио
if not os.path.exists('/content/piper-voice'):
    !git clone https://github.com/Holki9/piper-voice.git /content/piper-voice
else:
    !cd /content/piper-voice && git pull

# Копируем датасет в рабочую директорию
if os.path.exists('/content/piper-voice/dataset'):
    !rm -rf /content/dataset
    !cp -r /content/piper-voice/dataset /content/dataset
    print('✓ Датасет скопирован!')
    !ls -la /content/dataset/wavs | head -n 10

# Скачиваем базовую модель (Dmitri medium checkpoint)
!mkdir -p /content/base_model
!wget -q -c -O /content/base_model/dmitri.ckpt https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/ru/ru_RU/dmitri/medium/epoch%3D5589-step%3D1478840.ckpt
!wget -q -c -O /content/base_model/config.json https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/ru/ru_RU/dmitri/medium/config.json
print('✓ Базовая модель Dmitri medium загружена!')

## Шаг 3. Предобработка датасета (фонетизация espeak-ng)

In [ ]:
# Запуск предобработки
!/content/env/bin/python -m piper_train.preprocess \
    --language ru \
    --input-dir /content/dataset \
    --output-dir /content/training_dir \
    --dataset-format ljspeech \
    --single-speaker \
    --sample-rate 22050 \
    --max-workers 2

print('✓ Предобработка успешно завершена!')
!head -n 5 /content/training_dir/dataset.jsonl

## Шаг 4. Обучение (Fine-Tuning)

In [ ]:
import torch

# Определяем текущую эпоху чекпоинта
ckpt = torch.load('/content/base_model/dmitri.ckpt', map_location='cpu')
base_epoch = ckpt.get('epoch', 5589)
extra_epochs = 1000
target_max_epochs = base_epoch + extra_epochs
print(f'Базовая эпоха чекпоинта: {base_epoch}')
print(f'Целевая конечная эпоха max_epochs: {target_max_epochs} (+{extra_epochs} эпох дообучения)')

# Запуск обучения
!/content/env/bin/python -m piper_train \
    --dataset-dir /content/training_dir \
    --accelerator gpu \
    --devices 1 \
    --batch-size 8 \
    --validation-split 0.0 \
    --num-test-examples 0 \
    --max_epochs {target_max_epochs} \
    --checkpoint-epochs 50 \
    --precision 32 \
    --quality medium \
    --resume_from_checkpoint /content/base_model/dmitri.ckpt

## Шаг 5. Экспорт в ONNX и тестирование голоса

In [ ]:
import glob, os, shutil
from IPython.display import Audio, display

# Поиск последнего сохраненного чекпоинта
ckpts = sorted(glob.glob('/content/training_dir/lightning_logs/**/checkpoints/*.ckpt', recursive=True), key=os.path.getmtime)
if not ckpts:
    ckpts = sorted(glob.glob('/content/training_dir/*.ckpt'), key=os.path.getmtime)

best_ckpt = ckpts[-1]
print(f'Экспорт финального чекпоинта: {best_ckpt}')

# Экспорт в ONNX
!/content/env/bin/python -m piper_train.export_onnx {best_ckpt} /content/my_voice.onnx
shutil.copy('/content/training_dir/config.json', '/content/my_voice.onnx.json')

print('✓ Модель успешно экспортирована!')
!ls -lh /content/my_voice.onnx*

# Тестовая генерация речи
test_text = 'Привет! Это мой персональный дикторский голос. Теперь я говорю полностью офлайн прямо на вашем устройстве.'
!echo '{test_text}' | /content/env/bin/piper --model /content/my_voice.onnx --config /content/my_voice.onnx.json --output_file /content/test.wav

print('Слушайте результат синтеза:')
display(Audio('/content/test.wav'))

# Скачивание файлов модели
from google.colab import files
print('Скачивание модели...')
files.download('/content/my_voice.onnx')
files.download('/content/my_voice.onnx.json')